In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib.job_manager import load_config, split_config

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")

### Transform 

In [0]:
df_brand = spark.sql(f"""
    SELECT 
         _c0 AS ARTICLE_NBR
        ,_c1 AS CASE_EXPRESSION
        ,_c2 AS BRAND
    FROM 
        {bronze_master_brand}
""").dropDuplicates()

df_brand.createOrReplaceTempView("source")

In [0]:
validations.validate_table(
        spark, "intermediate", 'brand', config_validation, df_brand, stats_etl_path
    )

### Merge

In [0]:
df_brand.write.mode("overwrite").saveAsTable(silver_master_brand)

if archive_flag:
    save_archive(df_brand, silver_master_brand_archive, run_as_date)